# Experiment 2: put the constraint where the test settings live

The interval identity constrains the model's **own conditionals**. It needs no labels, so it is valid on any
trajectory distribution at all. That is the one structural advantage a self-consistency loss has over a
supervised one -- and the previous sweep threw it away, drawing the consistency batch from the same
random-walk dataset the MC loss already covers.

Which is plausibly why the value head got worse exactly where it was already weakest. `value_kl/bin 10`,
`bin 11` and `best far` read the value head on prefixes from the **conditioned** process: near-optimal paths
toward the goal, which a uniform random walk almost never produces. Both the MC supervision and the
consistency constraint were concentrated somewhere else entirely.

This notebook varies only **where the consistency rollouts come from**:

| source | rollouts | cost |
| --- | --- | --- |
| `data` | uniform from the random-walk training set -- the previous sweep's behaviour | free |
| `nearopt` | the same data, resampled to far starts that were near-optimal from their own start | free |
| `model` | the model's own R-conditioned rollouts, buffered and refreshed during training | ~200 forward passes per refresh |

**R is relabelled to the bin each rollout actually achieved, never the bin it was asked for.** Conditioning
the loss on a requested outcome the trajectory did not reach would impose the identity on a pair the model
may consider impossible, where `q_n(R)` is ~0 and the log form blows up -- the support problem in note
section 7. Asking for a high bin only shapes *which states get visited*; the label follows the outcome.

`ask="high"` draws requested bins from the top third, using no privileged information. `ask="best"` uses
`maze.best_bin` (the DP's answer) and is an oracle upper bound, not a deployable method -- run it to see how
much headroom the state distribution is worth, and read it as a ceiling.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}

In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
# Rebuild the dataset (100k random walks) and the exact test set from data/canonical/maze.txt. Deterministic.
!python run.py dataset | tail -4
!python run.py testset | tail -1

## 1. The loss configurations and the rollout sources

`SPECS` maps a run name to `(LossConfig, sampler factory)`. A factory gets `(model, tok, maze, data, n_train)`
and returns `cons_sampler(params, rng, n, step)` -- `train()` calls it once per step for the consistency batch.
Edit any of these; they are ordinary closures.

In [ ]:
from dataclasses import replace
from maze_consistency.train import LossConfig
from maze_consistency.evaluate import make_action_logits, rollout
from maze_consistency.testset import FAR
import maze_consistency.consistency as C
import numpy as np

OBJECTIVE, LAMBDA, CONS_BATCH = "local", 0.15, 16
BUFFER_N, REFRESH_EVERY = 128, 250


def data_sampler(model, tok, maze, data, n_train):
    """Uniform from the random-walk training set: what the previous sweep did."""
    def sample(params, rng, n, step):
        return C.rollout_batch(tok, maze, data, rng.integers(0, n_train, n))
    return sample


def nearopt_sampler(beta=1.0, far_only=True):
    """Same data, resampled toward rollouts that were near-optimal FROM THEIR OWN START:
    weight ~ exp(-beta * (L - dist[start])), restricted to far starts.

    Weighting by outcome bin instead is a trap. A high bin means a fast arrival, and only a near start can
    arrive fast, so bin-weighting collapses the batch onto near starts -- measured: mean start distance 4.0
    vs 11.5 for the full set, far fraction 0.08 vs 0.67. That is the opposite of the `best far` setting.
    Excess-over-optimal with far_only=True gives mean start distance 11.9, far fraction 1.00, mean excess 8.1
    steps, every rollout arriving. 8% of far-start random walks do reach the goal, which is enough to sample.

    Uses maze.dist (the shortest-path map, i.e. the environment layout) -- not the DP's policy or values."""
    def factory(model, tok, maze, data, n_train):
        L = data["length"][:n_train].astype(np.int64)
        reached = data["reached"][:n_train]
        dist = maze.dist[data["positions"][:n_train, 0].astype(np.int64)]
        excess = np.where(reached, L - dist, 10 ** 6)
        w = np.exp(-beta * np.minimum(excess, 500)).astype(np.float64)
        if far_only:
            w = w * (dist >= FAR)
        assert w.sum() > 0, "no rollouts match; loosen far_only or beta"
        w /= w.sum()
        def sample(params, rng, n, step):
            return C.rollout_batch(tok, maze, data, rng.choice(n_train, n, p=w))
        return sample
    return factory


def model_sampler(buffer_n=BUFFER_N, refresh_every=REFRESH_EVERY, ask="high"):
    """The model's own R-conditioned rollouts, refreshed every `refresh_every` steps.

    ask="high"  requested bins from the top third -- no privileged information
    ask="best"  each start's own optimal bin from the DP -- an ORACLE ceiling, not a method
    ask="any"   uniform over all bins
    """
    def factory(model, tok, maze, data, n_train):
        action_logits = make_action_logits(model, tok)
        state = {"buf": None, "last": -10**9}

        def refresh(params, rng):
            starts = rng.choice(maze.start_cells, buffer_n)
            if ask == "best":
                bins = maze.best_bin(starts)
            elif ask == "high":
                bins = rng.integers(2 * maze.K // 3, maze.K, buffer_n)
            else:
                bins = rng.integers(0, maze.K, buffer_n)
            state["buf"] = rollout(params, action_logits, tok, maze, bins, starts, rng)

        def sample(params, rng, n, step):
            if step - state["last"] >= refresh_every:
                refresh(params, rng)
                state["last"] = step
            # rollout_batch relabels R from length/reached, i.e. the bin ACHIEVED, not the one asked for
            return C.rollout_batch(tok, maze, state["buf"], rng.integers(0, buffer_n, n))
        return sample
    return factory


cons = LossConfig(mc=True, cons=True, cons_loss=OBJECTIVE, w_cons=LAMBDA, cons_batch=CONS_BATCH)
SPECS = {
    "mc":          (LossConfig(mc=True), None),              # no consistency term at all
    "data":        (cons, data_sampler),                     # the previous sweep
    "nearopt":     (cons, nearopt_sampler(beta=1.0, far_only=True)),
    "model_high":  (cons, model_sampler(ask="high")),
    "model_best":  (cons, model_sampler(ask="best")),        # oracle ceiling
}
LOSSES = {k: v[0] for k, v in SPECS.items()}                 # the plots below key off LOSSES

for k, (lc, f) in SPECS.items():
    print(f"  {k:<12} cons={lc.cons!s:<5} lambda={lc.w_cons if lc.cons else 0:<6g} "
          f"source={'-' if f is None else ('data' if f is data_sampler else 'custom')}")

## 2. The sweep

Same loop as `consistency_sweep.ipynb`: every parameter is an argument, edit the body in place.

In [ ]:
import os
import numpy as np
from maze_consistency.dataset import load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.train import train, load_run, RUNS_DIR, N_HELDOUT

MAZE, DATA = load_data()
TOK = Tokenizer(MAZE)
N_TRAIN = len(DATA["length"]) - N_HELDOUT
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)


def run_sweep(specs, seeds=(0,), steps=3000, batch=32, lr=1e-3, d_model=64, n_layers=2, n_heads=4,
              eval_every=250, eval_per_setting=50, heldout=HELDOUT, log_every=100, prefix="offdist",
              skip_existing=True, log=print):
    """specs: {run name: (LossConfig, sampler factory or None)}. The factory is called with
    (model, tok, maze, data, n_train) and returns cons_sampler(params, rng, n, step)."""
    ts = load_testset()
    rows = stratified_rows(ts, eval_per_setting, seed=0)
    cfg = ModelConfig.for_tokenizer(TOK, d_model=d_model, n_layers=n_layers, n_heads=n_heads)
    model = MazeTransformer(cfg)
    cons_eval = C.make_heldout_eval(model, TOK, MAZE, DATA, heldout)

    def eval_fn(params, fwd):
        m = score(params, fwd, TOK, ts, rows)
        m.pop("per_setting")
        m.update(cons_eval(params))
        return m

    done = {}
    for name, (lc, factory) in specs.items():
        for seed in seeds:
            run = f"{prefix}/{name}_s{seed}"
            if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
                log(f"[skip] {run} exists")
                continue
            sampler = factory(model, TOK, MAZE, DATA, N_TRAIN) if factory else None
            done[run] = train(name=run, steps=steps, batch=batch, lr=lr, d_model=d_model, n_layers=n_layers,
                              n_heads=n_heads, seed=seed, loss=lc, eval_fn=eval_fn, eval_every=eval_every,
                              log_every=log_every, log=log, cons_sampler=sampler)
    return done


PREFIX = "offdist"
run_sweep(SPECS, seeds=(0,), steps=3000, batch=32, eval_every=250, eval_per_setting=50, prefix=PREFIX)

## 3. Exact-test metrics

`act_kl` is KL(true action distribution || model), `value_kl` the same for the value head read in NOR mode.
Note `act_kl/NOR` is a near-trivial target -- the true NOR policy is uniform 1/4 -- so read the conditioned
settings (`bin 10`, `bin 11`, `best far`) as the real ones.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    """{run name: [history per seed]}. which="test" -> exact-test metrics at each checkpoint;
    which="train" -> logged loss parts, including cons / cond_gap / info_gain."""
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                out.setdefault(d.rsplit("_s", 1)[0], []).append(json.load(f)[which])
    return out


def plot_curves(prefix, metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"), logy=True, order=None, figsize=(4.2, 3.4)):
    runs = load_history(prefix, "test")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(len(metrics), len(settings),
                           figsize=(figsize[0] * len(settings), figsize[1] * len(metrics)), squeeze=False)
    for i, metric in enumerate(metrics):
        for j, setting in enumerate(settings):
            a, key = ax[i, j], f"{metric}/{setting}"
            for c, name in enumerate(names):
                hists = runs[name]
                steps = [m["step"] for m in hists[0]]
                ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
                a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
                if len(hists) > 1:
                    a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
            if logy:
                a.set_yscale("log")
            a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=8)
    fig.tight_layout()
    return fig


plot_curves(PREFIX)
plt.show()

## 4. Collapse diagnostics

`cond_gap` -> 0 means the conditioned and unconditioned policies have merged; `info_gain` -> 0 means the
reward head has gone flat. Either decaying while `cons` falls = lambda too high.

In [ ]:
def plot_diagnostics(prefix, keys=(("tf", "data: teacher-forced next-token loss", True),
                                   ("cons", "consistency objective", True),
                                   ("cond_gap", "cond_gap: mean |v_t - u_t|   (-> 0 = R ignored)", False),
                                   ("info_gain", "info_gain: log q_n(R) - log q_0(R)   (-> 0 = head flat)", False)),
                     order=None, figsize=(4.6, 3.8)):
    """Train-side view. keys is (history key, panel title, log y). Configs with no consistency term simply
    do not appear in the cons/cond_gap/info_gain panels."""
    runs = load_history(prefix, "train")
    names = [n for n in (order or LOSSES) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, (key, title, logy) in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name)
            if len(hists) > 1:
                a.fill_between(steps, np.nanmin(ys, 0), np.nanmax(ys, 0), color=f"C{c}", alpha=.15)
        a.set_yscale("log") if logy else a.axhline(0, color="k", lw=.8, ls=":")
        a.set_title(title, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


plot_diagnostics(PREFIX)
plt.show()

## 5. Held-out consistency against the exact model

All six objectives on one fixed held-out batch, identical for every run. The true model scores 0.

In [ ]:
GT = compute_ground_truth(MAZE)
exact, _ = C.exact_losses(MAZE, GT, DATA, HELDOUT)
print("the true model on the same held-out rollouts (0 = perfectly consistent):")
print("  " + "  ".join(f"{k} {float(np.asarray(v).mean()):.2e}" for k, v in exact.items()))

plot_curves(PREFIX, metrics=("cons",), settings=tuple(C.ALL), figsize=(3.4, 3.2))
plt.show()
plot_curves(PREFIX, metrics=("diag",), settings=("cond_gap", "info_gain", "drift"), logy=False)
plt.show()

## 6. Final numbers

In [ ]:
def final_table(prefix, baseline="mc", metrics=("act_kl", "value_kl"),
                settings=("NOR", "bin 10", "bin 11", "best far"),
                diag=("cons/local", "cons/all_scaled", "cons/multiscale", "diag/cond_gap", "diag/info_gain"),
                order=None):
    test = load_history(prefix, "test")
    cols = [f"{m}/{s}" for m in metrics for s in settings]
    names = [n for n in (order or LOSSES) if n in test] or list(test)
    mean_last = lambda hs, k: float(np.mean([h[-1].get(k, np.nan) for h in hs]))
    rows = {n: {k: mean_last(test[n], k) for k in cols + list(diag)} for n in names}
    base = rows.get(baseline)
    w = max(len(n) for n in rows) + 2

    print(f"exact-test KL in nats, lower is better. (d) = minus `{baseline}`, so negative beats baseline.")
    for m in metrics:
        group = [f"{m}/{s}" for s in settings]
        print()
        print(" " * w + "".join(f"{k.split('/')[1]:>22}" for k in group))
        print(f"{m:<{w}}" + "".join(f"{'value':>12}{'(d)':>10}" for _ in group))
        for n, r in rows.items():
            line = "".join(f"{r[k]:12.4f}" + ("".rjust(10) if base is None or n == baseline
                                              else f"{r[k] - base[k]:+10.4f}") for k in group)
            print(f"{n:<{w}}" + line)
    print()
    print("held out, same rollouts for every run; the true model scores 0 on every cons/ column")
    print(f"{'config':<{w}}" + "".join(f"{k:>18}" for k in diag))
    for n, r in rows.items():
        print(f"{n:<{w}}" + "".join(f"{r[k]:18.4f}" if np.isfinite(r[k]) else f"{'-':>18}" for k in diag))
    return rows


_ = final_table(PREFIX)

## 7. Where did the rollouts actually go?

The experiment only means something if the three sources really do visit different states. This compares the
consistency batches each sampler produces from the trained baseline: outcome-bin distribution and start
distance to the goal. If `model_high` looks like `data`, the model is not yet good enough for its own
rollouts to be off-distribution, and a null result says nothing about the idea.

In [ ]:
import matplotlib.pyplot as plt
from maze_consistency.testset import FAR

def source_distributions(prefix=PREFIX, run="mc", n=512, seed=0):
    params, mcfg = load_run(f"{prefix}/{run}_s{seed}")
    model = MazeTransformer(mcfg)
    rng = np.random.default_rng(seed)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for c, (name, factory) in enumerate([("data", data_sampler), ("nearopt", nearopt_sampler()),
                                         ("model_high", model_sampler(buffer_n=n, ask="high")),
                                         ("model_best", model_sampler(buffer_n=n, ask="best"))]):
        s = factory(model, TOK, MAZE, DATA, N_TRAIN)
        b = s(params, rng, n, 0)
        bins = np.asarray(b["R_bin"])
        starts = np.asarray(b["x_nor"][:, 1, 1]) + np.asarray(b["x_nor"][:, 1, 2]) * MAZE.W
        ax[0].hist(bins, bins=np.arange(MAZE.K + 1) - .5, histtype="step", lw=1.6, label=name)
        ax[1].hist(MAZE.dist[starts], bins=np.arange(MAZE.dist.max() + 2) - .5, histtype="step", lw=1.6,
                   label=name)
        print(f"  {name:<12} reached {float((bins > 0).mean()):.2f}   mean bin {bins.mean():5.2f}   "
              f"mean n {float(np.asarray(b['lengths']).mean()):6.1f}   "
              f"start dist {MAZE.dist[starts].mean():5.2f}   far frac {float((MAZE.dist[starts] >= FAR).mean()):.2f}")
    ax[0].set_yscale("log"); ax[0].set_xlabel("outcome bin of the consistency batch"); ax[0].legend(fontsize=8)
    ax[1].set_xlabel("start distance to goal"); ax[1].legend(fontsize=8)
    for a in ax:
        a.grid(alpha=.3)
    fig.tight_layout()
    return fig


source_distributions()
plt.show()